# ResNet50 ImageNet Training on Kaggle

This notebook trains ResNet50 on the full ImageNet-1K dataset (1000 classes).

**Full Training**: Trains on entire ImageNet dataset for 90 epochs with standard settings.

## Requirements:
1. **Dataset**: `imagenet-object-localization-challenge` (add to notebook inputs)
2. **GPU**: Enable GPU accelerator (required for reasonable training time)
3. **Internet**: Enable for package installation
4. **Files**: Upload required Python files (see setup section)

## Features:
- ✅ ResNet50 (25.5M parameters)
- ✅ Full ImageNet training (1.28M training images)
- ✅ Strong data augmentation
- ✅ Reuses code from common/ directory
- ✅ SGD optimizer with momentum and step LR decay
- ✅ Automatic checkpointing and history tracking
- ✅ Early stopping support

## Training Configuration:
- **Batch size**: 256
- **Epochs**: 90
- **Learning rate**: 0.1 (decay by 0.1 every 30 epochs)
- **Optimizer**: SGD with Nesterov momentum (0.9)
- **Weight decay**: 1e-4
- **Expected training time**: ~7-10 days on single GPU

## 1. Environment Setup

**Important**: This notebook requires the following files to be uploaded to Kaggle:
1. `assignment9/model.py` - ResNet50 model definition
2. `assignment9/data.py` - ImageNet data loading
3. `common/trainer.py` - Generic training loop
4. `common/utils.py` - Utility functions

Upload these files to `/kaggle/working/` before running the notebook.

In [ ]:
import os
import sys

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    print("✅ Kaggle environment")
    IMAGENET_PATH = '/kaggle/input/imagenet-object-localization-challenge'
    WORKING_DIR = '/kaggle/working'
    ASSIGNMENT_DIR = '/kaggle/working'
else:
    IMAGENET_PATH = './data/imagenet'
    WORKING_DIR = '.'
    ASSIGNMENT_DIR = os.path.dirname(os.path.abspath('__file__'))
    
# Add to Python path for imports
if ASSIGNMENT_DIR not in sys.path:
    sys.path.insert(0, ASSIGNMENT_DIR)
parent_dir = os.path.dirname(ASSIGNMENT_DIR)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"ImageNet Path: {IMAGENET_PATH}")
print(f"Working Dir: {WORKING_DIR}")
print(f"Python Path: {sys.path[:3]}")

In [ ]:
%pip install -q albumentations opencv-python-headless
print("✅ Packages installed")

## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
import json

# Import common utilities
from common.utils import (
    get_device, set_random_seed, 
    plot_training_history, save_training_info
)
from common.trainer import create_trainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
print("✅ Imported common utilities")

## 3. Define ResNet50 Model

In [ ]:
# Import ResNet50 from model.py
from assignment9.model import ResNet50ImageNet, resnet50

print("✅ Imported ResNet50 from assignment9.model")

## 4. Define Data Loading

In [ ]:
# Import ImageNet data classes from data.py
from assignment9.data import ImageNetDataset, get_imagenet_data_loaders

print("✅ Imported ImageNetDataset from assignment9.data")

## 5. Configuration

In [ ]:
config = {
    'batch_size': 256,  # Larger batch size for full training
    'num_workers': 4,
    'lr': 0.1,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'num_classes': 1000,
    'seed': 42,
    'epochs': 90,  # Standard ImageNet training
    'scheduler': 'step',
    'step_size': 30,
    'gamma': 0.1,
    'early_stopping_patience': 10,
    'checkpoint_dir': os.path.join(WORKING_DIR, 'checkpoints'),
}

# Set random seed
set_random_seed(config['seed'])

# Get device
device = get_device()

print(f"Device: {device}")
print(f"\nConfiguration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 6. Load Data

In [ ]:
print("📥 Loading ImageNet dataset...")
print(f"  Path: {IMAGENET_PATH}")

# Load full ImageNet dataset (no sample limit for full training)
train_loader, val_loader = get_imagenet_data_loaders(
    data_dir=IMAGENET_PATH,
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    augment=True,
    pin_memory=torch.cuda.is_available(),
    limit_samples=None  # Full dataset
)

print(f"\n✅ Data loaded successfully")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Train samples: ~{len(train_loader) * config['batch_size']:,}")
print(f"  Val samples: ~{len(val_loader) * config['batch_size']:,}")

## 7. Create Model

In [ ]:
print("\n🏗️  Creating ResNet50 model...")

# Create model using the function from model.py
model = resnet50(num_classes=config['num_classes'], dropout=0.0, pretrained=False)
model = model.to(device)

# Count parameters
params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Information:")
print(f"  Total parameters: {params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {params * 4 / 1e6:.1f} MB (float32)")

# Test forward pass
print(f"\n🧪 Testing forward pass...")
model.eval()
with torch.no_grad():
    test_input = torch.randn(2, 3, 224, 224).to(device)
    test_output = model(test_input)
    print(f"  Input shape: {test_input.shape}")
    print(f"  Output shape: {test_output.shape}")
model.train()

print("\n✅ Model created successfully")

## 8. Setup Training

In [ ]:
print("\n⚙️  Setting up training components...")

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.SGD(
    model.parameters(),
    lr=config['lr'],
    momentum=config['momentum'],
    weight_decay=config['weight_decay'],
    nesterov=True
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['step_size'],
    gamma=config['gamma']
)

print(f"  Loss: CrossEntropyLoss")
print(f"  Optimizer: SGD")
print(f"    - Learning rate: {config['lr']}")
print(f"    - Momentum: {config['momentum']}")
print(f"    - Weight decay: {config['weight_decay']}")
print(f"    - Nesterov: True")
print(f"  Scheduler: StepLR")
print(f"    - Step size: {config['step_size']} epochs")
print(f"    - Gamma: {config['gamma']}")

print("\n✅ Training components ready")

## 9. Train for 1 Batch

In [ ]:
print("\n" + "="*70)
print("🚀 Starting Full ImageNet Training")
print("="*70)
print(f"  Epochs: {config['epochs']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Learning rate: {config['lr']}")
print(f"  Device: {device}")
print("="*70 + "\n")

# Create trainer using common/trainer.py
trainer = create_trainer(
    model=model,
    device=device,
    train_loader=train_loader,
    test_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    config=config,
    l2_lambda=0.0  # Weight decay is handled by optimizer
)

# Train the model
print(f"⏰ Training started at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

training_history = trainer.train(
    num_epochs=config['epochs'],
    early_stopping_patience=config['early_stopping_patience'],
    min_delta=0.001,
    checkpoint_dir=config['checkpoint_dir'],
    scheduler_type='step',
    save_best=True,
    save_latest=True,
    verbose=True
)

print(f"\n⏰ Training completed at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*70)
print("✅ Training Complete!")
print("="*70)

## 10. Save Results

In [ ]:
print("\n📊 Generating training visualizations...")

# Plot training history
plot_path = os.path.join(WORKING_DIR, 'training_history_resnet50_imagenet.png')
plot_training_history(training_history, save_path=plot_path, show_plot=False)
print(f"  Training plot saved: {plot_path}")

# Get final evaluation
print("\n🧪 Final evaluation on validation set...")
final_test_loss, final_test_acc = trainer.evaluate()

# Prepare training summary
best_test_acc = max(training_history['test_accuracies']) if training_history['test_accuracies'] else final_test_acc
total_epochs = len(training_history['epochs']) if training_history['epochs'] else config['epochs']

print(f"\n📈 Training Summary:")
print(f"  Total epochs: {total_epochs}")
print(f"  Best validation accuracy: {best_test_acc:.2f}%")
print(f"  Final validation accuracy: {final_test_acc:.2f}%")
print(f"  Final validation loss: {final_test_loss:.4f}")

# Save comprehensive training info
model_info = {
    'model': 'ResNet50',
    'dataset': 'ImageNet-1K',
    'num_classes': config['num_classes'],
    'parameters': params,
    'final_val_accuracy': final_test_acc,
    'final_val_loss': final_test_loss,
    'best_val_accuracy': best_test_acc,
    'total_epochs': total_epochs,
    'config': config,
    'training_history': {
        'epochs': training_history['epochs'],
        'train_losses': training_history['train_losses'],
        'train_accuracies': training_history['train_accuracies'],
        'test_losses': training_history['test_losses'],
        'test_accuracies': training_history['test_accuracies'],
        'learning_rates': training_history['learning_rates']
    }
}

info_path = os.path.join(WORKING_DIR, 'resnet50_imagenet_training_info.json')
save_training_info(model_info, info_path)

# Save final model
final_model_path = os.path.join(WORKING_DIR, 'resnet50_imagenet_final.pth')
torch.save(model.state_dict(), final_model_path)
print(f"\n💾 Final model saved: {final_model_path}")

print("\n" + "="*70)
print("🎉 All training tasks completed successfully!")
print("="*70)

## Summary

✅ Successfully completed full ResNet50 ImageNet training:
- ResNet50 model (25.5M parameters)
- Full ImageNet-1K dataset (1.28M training images, 50K validation)
- Complete training for 90 epochs
- Automatic checkpointing (best + latest)
- Training history visualization
- Model saved for inference

### Training Results:
Check the output above for:
- Final validation accuracy (target: ~70-76%)
- Training curves (loss and accuracy over time)
- Best checkpoint location
- Training duration

### Saved Files:
- `resnet50_imagenet_final.pth` - Final model weights
- `checkpoints/best_model.pth` - Best model checkpoint
- `checkpoints/latest_checkpoint.pth` - Latest checkpoint
- `training_history_resnet50_imagenet.png` - Training curves
- `resnet50_imagenet_training_info.json` - Full training statistics

### Next Steps:
1. Evaluate on test set
2. Implement Top-5 accuracy metric
3. Try mixed precision training (AMP) for faster training
4. Fine-tune on downstream tasks
5. Export to ONNX for deployment